# Minggu 3-4 — RAG Pipeline Runner (Kaggle GPU)

Sama seperti notebook Minggu 2: notebook ini **hanya menjalankan** `src/rag_pipeline.py`,
logika inti tetap di repo GitHub.

**Sebelum menjalankan, lihat `docs/TROUBLESHOOTING_KAGGLE.md`** -- terutama soal
Llama-3.2 adalah **gated model** (butuh HF token), ini yang paling sering bikin stuck.

**Checklist sebelum run:**
1. GPU aktif (Settings -> Accelerator -> GPU)
2. Internet aktif (Settings -> Internet -> On)
3. Secret `HF_TOKEN` sudah ditambahkan (Add-ons -> Secrets)
4. `data/index/anime.index` dan `id_mapping.pkl` dari Minggu 2 sudah diupload sebagai Kaggle Dataset input
5. Sudah accept license Llama-3.2-3B-Instruct di huggingface.co


In [ ]:
!git clone https://github.com/Rahmat-Ramadhan15/Anime-RAG-Recommender.git
%cd Anime-RAG-Recommender


In [ ]:
!pip install -q sentence-transformers faiss-gpu transformers accelerate langchain-core pyyaml bitsandbytes


In [ ]:
# Login Hugging Face (WAJIB untuk Llama-3.2 -- gated model)
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

login(UserSecretsClient().get_secret("HF_TOKEN"))


In [ ]:
# Salin index hasil Minggu 2 (sesuaikan nama dataset Kaggle Anda)
import shutil, os

os.makedirs("data/index", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

shutil.copy("/kaggle/input/anime-index-week2/anime.index", "data/index/anime.index")
shutil.copy("/kaggle/input/anime-index-week2/id_mapping.pkl", "data/index/id_mapping.pkl")
shutil.copy("/kaggle/input/anime-documents-week1/anime_documents.jsonl", "data/processed/anime_documents.jsonl")
print("Semua file siap.")


In [ ]:
import sys
sys.path.append("src")

from rag_pipeline import RagPipeline

pipe = RagPipeline()
pipe.load_index()
pipe.load_llm(quantize=True)  # 4-bit -- hemat VRAM, cocok untuk GPU T4 Kaggle


In [ ]:
# Uji satu query pada ketiga kondisi ablation sekaligus (Kondisi C: enrichment_data masih placeholder kosong,
# akan disambungkan setelah src/enrichment.py selesai di Minggu 5)

query = "Rekomendasikan anime action dengan tema luar angkasa"

for cond_name, kwargs in [
    ("A", {"use_retrieval": False}),
    ("B", {"use_retrieval": True, "use_enrichment": False}),
    ("C", {"use_retrieval": True, "use_enrichment": True, "enrichment_data": {}}),
]:
    result = pipe.generate(query, **kwargs)
    print(f"=== Kondisi {cond_name} ===")
    print(result["answer"])
    print()


## Langkah selanjutnya

- Simpan hasil percobaan (query + jawaban tiap kondisi) sebagai Kaggle Notebook Output untuk dianalisis nanti
- Lanjut ke Minggu 4: uji top-k (3/5/10) dengan memanggil `pipe.generate(query, k=...)` untuk beberapa nilai k,
  bandingkan kualitas retrieval sebelum menetapkan k final di `configs/config.yaml`
- Setelah `src/enrichment.py` selesai (Minggu 5), isi `enrichment_data` dengan hasil panggilan Jikan API
  yang sesungguhnya untuk menjalankan Kondisi C secara penuh
